# PACSUN2603 Data
This notebook unifies everything into one dataframe for each experiment to make plotting easier later.

In [63]:
import pandas as pd
import pytz
from pvlib.solarposition import get_solarposition
import time

## Geographic Data

In [55]:
# Read the CSV data directly from the file.
# The 'comment='#' will automatically skip lines starting with '#'.
gdf = pd.read_csv("ISAC_PACSUN2603.csv", comment='#')

# Convert 'iso_time' column to datetime objects
gdf['Datetime'] = pd.to_datetime(gdf['DeviceDateTime'], unit = 's').dt.tz_localize('Pacific/Tongatapu')
gdf['Datetime'] = gdf['Datetime'].astype('datetime64[s, Pacific/Tongatapu]')
# Sort the DataFrame by time to ensure correct animation order
gdf = gdf.sort_values(by='Datetime')
gdf.head()

,DataId,CommId,DeviceName,DeviceDateTime,AgeInSeconds,BatteryVoltage,GpsQuality,Latitude,Longitude,SubmergedBoolean,Temperature0cm,Unnamed: 11,station,Datetime
386,66038667,3.010000e+14,USC/MTEL-UT-0001,3/5/26 17:45,63906,7.5,3,33.204360,-117.306144,0,-0.05,NaN,387,2026-03-05 17:45:00+13:00
385,66038669,3.010000e+14,USC/MTEL-UT-0001,3/5/26 17:50,64206,12.0,3,33.204357,-117.306149,0,NaN,NaN,386,2026-03-05 17:50:00+13:00
384,66038680,3.010000e+14,USC/MTEL-UT-0001,3/5/26 17:55,64507,12.0,3,33.204357,-117.306141,0,NaN,NaN,385,2026-03-05 17:55:00+13:00
383,66038795,3.010000e+14,USC/MTEL-UT-0001,3/5/26 18:00,64808,12.0,3,33.204363,-117.306149,0,NaN,NaN,384,2026-03-05 18:00:00+13:00
382,66038808,3.010000e+14,USC/MTEL-UT-0001,3/5/26 18:05,65108,12.0,3,33.204357,-117.306144,0,NaN,NaN,383,2026-03-05 18:05:00+13:00


## ISAC1

In [83]:
df.Datetime[1]

Timestamp('2026-03-19 09:32:55+1300', tz='Pacific/Tongatapu')

In [89]:
# Unify our tables....
dfs = []

for filename in ["ISAC1_top"]:
    df = pd.read_csv(filename+".TXT", skiprows = 4)
    df.columns = ['unixTime','background','ambient', 'backscatter', 'pressure','waterTemp','battery']

    # Change these lines to filter as you'd like:
    df['backscatter_filtered'] = df['backscatter'].rolling(window=20, center=True).median()
    df['ambient_filtered'] = df['ambient'].rolling(window=11, center=True).median()

    df['Datetime'] = pd.to_datetime(df['unixTime'], unit='s').dt.tz_localize('Pacific/Tongatapu') #
    # df = df.sort_values(by="unixTime")
    test_val = df.unixTime[1]
    print(f"Time check: Data collection starts at Unix timestamp  {test_val}")
    print(f"Standard UTC conversion: {time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime(test_val))}")
    print(f"Local time in Tonga:     {df.Datetime.iloc[1]}")
    # Get S/N....
    with open(filename+".TXT", 'r') as file:
        lines = file.readlines()
        if len(lines) > 1:
            # Get the second line and strip the trailing newline character
            SN = lines[1].strip('\n')
            print(SN)
        else:
            print("File has fewer than two lines.")
    

    suffix = "_" +SN[11:15]
    df.columns = [
        f"{col}{suffix}" if col != 'Datetime' else col 
        for col in df.columns
    ]
    dfs.append(df)

# Concatenate all the things:
df = pd.concat(dfs, ignore_index=True)

# Calculate solar zenith angle:
times = df['Datetime']
solpos = get_solarposition(times, -21, -175) # lat, lon
df['zenith'] = solpos['zenith'].values
df['elevation'] = solpos['elevation'].values
df['azimuth'] = solpos['azimuth'].values
# Filter to time range of experiment:
# df = df[df['Datetime'].between('2026-03-21', '2026-03-23')]
# df = df.sort_values('Datetime') # sort to avoid plotting errors

# Sanity check: This should be noonish.
peak_sun = df.loc[df['zenith'].idxmin()]
print(f"Sun is highest at: {peak_sun['Datetime']}")

# Add GPS information
# df = df.sort_values('Datetime')
# gdf = gdf.sort_values('Datetime')
# df['Datetime'] = pd.to_datetime(df['Datetime'])
df = pd.merge_asof(df, gdf, on='Datetime', direction='backward')

# Save dataframe
df.to_csv('ISAC1.csv', index=False)

Time check: Data collection starts at Unix timestamp  1773912775
Standard UTC conversion: 2026-03-19 09:32:55
Local time in Tonga:     2026-03-19 09:32:55+13:00
OpenOBS SN:123
Sun is highest at: 2026-03-19 09:35:41+13:00


## ISAC2

In [57]:
# Unify our tables....
dfs = []

for filename in ["ISAC2_mid_123"]:
    df = pd.read_csv(filename+".TXT", skiprows = 4)
    df.columns = ['unixTime','background','ambient', 'backscatter', 'pressure','waterTemp','battery']

    # Change these lines to filter as you'd like:
    df['backscatter_filtered'] = df['backscatter'].rolling(window=20, center=True).median()
    df['ambient_filtered'] = df['ambient'].rolling(window=11, center=True).median()

    df['Datetime'] = pd.to_datetime(df['unixTime'], unit='s').dt.tz_localize('Pacific/Tongatapu') #
    df = df.sort_values(by="unixTime")

    # Get S/N....
    with open(filename+".TXT", 'r') as file:
        lines = file.readlines()
        if len(lines) > 1:
            # Get the second line and strip the trailing newline character
            SN = lines[1].strip('\n')
            print(SN)
        else:
            print("File has fewer than two lines.")
    

    suffix = "_" +SN[11:15]
    df.columns = [
        f"{col}{suffix}" if col != 'Datetime' else col 
        for col in df.columns
    ]
    dfs.append(df)

# Concatenate all the things:
df = pd.concat(dfs, ignore_index=True)

# Localize datetime to Tonga, assuming computer RTC is in Pacific Time:
# df['Datetime'] = df['Datetime'].dt.tz_localize('Pacific/Tongatapu')

# Calculate solar zenith angle:
times = df['Datetime']
solpos = get_solarposition(times, -21, -175) # lat, lon
df['zenith'] = solpos['zenith'].values
df['elevation'] = solpos['elevation'].values
df['azimuth'] = solpos['azimuth'].values
# Filter to time range of experiment:
# df = df[df['Datetime'].between('2026-03-21', '2026-03-23')]
# df = df.sort_values('Datetime') # sort to avoid plotting errors

# Sanity check: This should be noonish.
peak_sun = df.loc[df['zenith'].idxmin()]
print(f"Sun is highest at: {peak_sun['Datetime']}")

# Add GPS information
df = df.sort_values('Datetime')
gdf = gdf.sort_values('Datetime')
df = pd.merge_asof(df, gdf, on='Datetime', direction='backward')

# Save dataframe
df.to_csv('ISAC2.csv', index=False)

OpenOBS SN:123
Sun is highest at: 2026-03-21 12:47:12+13:00


## ISAC 3


In [58]:
# Unify our tables....
dfs = []

for filename in ["ISAC3_top_123", "ISAC3_mid_117", "ISAC3_mid_113"]:
    df = pd.read_csv(filename+".TXT", skiprows = 4)
    df.columns = ['unixTime','background','ambient', 'backscatter', 'pressure','waterTemp','battery']

    # Change these lines to filter as you'd like:
    df['backscatter_filtered'] = df['backscatter'].rolling(window=20, center=True).median()
    df['ambient_filtered'] = df['ambient'].rolling(window=11, center=True).median()

    df['Datetime'] = pd.to_datetime(df['unixTime'], unit='s').dt.tz_localize('Pacific/Tongatapu') #
    df = df.sort_values(by="unixTime")

    # Get S/N....
    with open(filename+".TXT", 'r') as file:
        lines = file.readlines()
        if len(lines) > 1:
            # Get the second line and strip the trailing newline character
            SN = lines[1].strip('\n')
            print(SN)
        else:
            print("File has fewer than two lines.")
    

    suffix = "_" +SN[11:15]
    df.columns = [
        f"{col}{suffix}" if col != 'Datetime' else col 
        for col in df.columns
    ]
    dfs.append(df)

# Concatenate all the things:
print('Concatenating:')
print(dfs)
df = pd.concat(dfs, ignore_index=True)

# Localize datetime to Tonga, assuming computer RTC is in Pacific Time:
# df['Datetime'] = df['Datetime'].dt.tz_localize('Pacific/Tongatapu')

# Calculate solar zenith angle:
times = df['Datetime']
solpos = get_solarposition(times, -21, -175) # lat, lon
df['zenith'] = solpos['zenith'].values
df['elevation'] = solpos['elevation'].values
df['azimuth'] = solpos['azimuth'].values
# Filter to time range of experiment:
# df = df[df['Datetime'].between('2026-03-21', '2026-03-23')]
df = df.sort_values('Datetime') # sort to avoid plotting errors

# Sanity check: This should be noonish.
peak_sun = df.loc[df['zenith'].idxmin()]
print(f"Sun is highest at: {peak_sun['Datetime']}")

# Add GPS information
df = df.sort_values('Datetime')
gdf = gdf.sort_values('Datetime')
df = pd.merge_asof(df, gdf, on='Datetime', direction='backward')

# Save dataframe
df.to_csv('ISAC3.csv', index=False)


OpenOBS SN:123
OpenOBS SN:117
OpenOBS SN:113
Concatenating:
[         unixTime_123  background_123  ambient_123  backscatter_123  \
0          1774110676             122          172             8923   
1          1774110676             165          172             8917   
2          1774110676             208          178             8918   
3          1774110676             256          170             8919   
4          1774110676             299          174             8922   
...               ...             ...          ...              ...   
2707921    1774232176       121497944            7             8479   
2707922    1774232176       121497987            4             8480   
2707923    1774232176       121498032            3             8485   
2707924    1774232176       121498079            6             8482   
2707925    1774232176       121498122            5             8480   

         pressure_123  waterTemp_123  battery_123  backscatter_filtered_123  \
0      

In [92]:
# Unify our tables....
dfs = []

for filename in ["ISAC3_top_123", "ISAC3_mid_117", "ISAC3_mid_113"]:
    df = pd.read_csv(filename+".TXT", skiprows = 4)
    df.columns = ['unixTime','background','ambient', 'backscatter', 'pressure','waterTemp','battery']

    # Change these lines to filter as you'd like:
    df['backscatter_filtered'] = df['backscatter'].rolling(window=20, center=True).median()
    df['ambient_filtered'] = df['ambient'].rolling(window=11, center=True).median()

    df['Datetime'] = pd.to_datetime(df['unixTime'], unit='s').dt.tz_localize('Pacific/Tongatapu') #
    # df = df.sort_values(by="unixTime")
    test_val = df.unixTime[1]
    print(f"Time check for {filename}: Data collection starts at Unix timestamp  {test_val}")
    print(f"Standard UTC conversion: {time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime(test_val))}")
    print(f"Local time in Tonga:     {df.Datetime.iloc[1]}")
    # Get S/N....
    with open(filename+".TXT", 'r') as file:
        lines = file.readlines()
        if len(lines) > 1:
            # Get the second line and strip the trailing newline character
            SN = lines[1].strip('\n')
            print(SN)
        else:
            print("File has fewer than two lines.")


    suffix = "_" +SN[11:15]
    df.columns = [
        f"{col}{suffix}" if col != 'Datetime' else col
        for col in df.columns
    ]
    dfs.append(df)

# Concatenate all the things:
df = pd.concat(dfs, ignore_index=True)

# Calculate solar zenith angle:
times = df['Datetime']
solpos = get_solarposition(times, -21, -175) # lat, lon
df['zenith'] = solpos['zenith'].values
df['elevation'] = solpos['elevation'].values
df['azimuth'] = solpos['azimuth'].values
# Filter to time range of experiment:
# df = df[df['Datetime'].between('2026-03-21', '2026-03-23')]
# df = df.sort_values('Datetime') # sort to avoid plotting errors

# Sanity check: This should be noonish.
peak_sun = df.loc[df['zenith'].idxmin()]
print(f"Sun is highest at: {peak_sun['Datetime']}")

# Add GPS information
df = df.sort_values('Datetime')
# gdf = gdf.sort_values('Datetime')
# df['Datetime'] = pd.to_datetime(df['Datetime'])
df = pd.merge_asof(df, gdf, on='Datetime', direction='backward')

# Save dataframe
df.to_csv('ISAC3.csv', index=False)

Time check for ISAC3_top_123: Data collection starts at Unix timestamp  1774110676
Standard UTC conversion: 2026-03-21 16:31:16
Local time in Tonga:     2026-03-21 16:31:16+13:00
OpenOBS SN:123
Time check for ISAC3_mid_117: Data collection starts at Unix timestamp  1774100023
Standard UTC conversion: 2026-03-21 13:33:43
Local time in Tonga:     2026-03-21 13:33:43+13:00
OpenOBS SN:117
Time check for ISAC3_mid_113: Data collection starts at Unix timestamp  1774114955
Standard UTC conversion: 2026-03-21 17:42:35
Local time in Tonga:     2026-03-21 17:42:35+13:00
OpenOBS SN:113
Sun is highest at: 2026-03-22 12:46:54+13:00
